# FallVision 06: pack the dataset for submission

Copies **exactly the videos our splits use** (1,457 videos; + 20 own videos if `WITH_OWN = True`) from
the download notebooks' outputs into one clean folder, with the layout the code reads, the labels,
the split list, `DATA.md` and `manifest.csv` (size and MD5 of every video). Done by
`src/check_dataset.py`; nothing is changed in the videos.

**Before running (Kaggle)**
- Inputs: the outputs of `01_download_omnifall_videos` and `02_download_gmdcsa24`
  (+ the dataset `fallvision-own-videos` if `WITH_OWN = True`).
- *Settings*: Accelerator **None** (CPU is enough), Internet **on**. No secret needed.
- Time: about 15-30 minutes. **Save Version > Save & Run All (Commit)**.

**Afterwards:** the folder `FallVision_dataset` is in this notebook's Output. Share it with the
instructor (e.g. *Output > New Dataset*, private, then share it with the instructor's Kaggle account)
or download it. Le2i, CAUCAFall, MCFD and OOPS are research-only: never make it public.

In [ ]:
# ---- settings: the only cell you may need to change ----
REPO_URL = "https://github.com/Prish1509/project_fallvision.git"   # your repository (public)
BRANCH = "main"
REPO = "/kaggle/working/project_fallvision"   # where the code is cloned in this notebook
DEST = "/kaggle/working/FallVision_dataset"   # the packed dataset (kept in the Output)
WITH_OWN = False                # True only if all three members agree to share the own recordings

## 1. Code

In [ ]:
import os, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
import kaggle_helpers as kh
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## 2. Inputs

In [ ]:
OMNI = kh.find_omnifall()
GMD = kh.find_gmd()
assert OMNI and GMD, "add the outputs of the two data notebooks as inputs (see the top of this notebook)"
OWN = kh.find_own(skip=(OMNI, GMD)) if WITH_OWN else None
if WITH_OWN:
    assert OWN, "WITH_OWN is True but the own videos were not found: add the fallvision-own-videos dataset"

## 3. Check and pack

In [ ]:
kh.run([sys.executable, "src/check_dataset.py", "--omnifall-root", OMNI, "--gmd-root", GMD,
        *(["--own-root", OWN, "--with-own"] if OWN else []), "--pack-to", DEST])

In [ ]:
total = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(DEST) for f in fs)
print(f"{DEST}: {total / 1e9:.2f} GB")
print(sorted(os.listdir(DEST)))